In [4]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision
from torchvision.datasets import CIFAR10

In [5]:
#Datasets and DataLoader
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

#image -> scale(0,1) -> normalise -> (-1 , 1)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5 , 0.5 , 0.5) , (0.5, 0.5 , 0.5))
])

trainset = CIFAR10(root="./data", train=True , download=True , transform=transform)
testset = CIFAR10(root="./data", train=False , download=True , transform=transform)

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 170M/170M [25:43<00:00, 110kB/s]


In [6]:
trainloader = DataLoader(trainset , batch_size=64, shuffle=True)
testloader  = DataLoader(testset , batch_size=64)

# Build the CNN

In [7]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN , self).__init__() # super constructor


        self.conv_layers = nn.Sequential(
            # 1st convolution layer
            nn.Conv2d(3 ,32 ,kernel_size=3 , padding=1), # input channels , no of kernels applied generally multiple of 2 , kernel_size , padding 
            nn.ReLU(),
            nn.MaxPool2d(2 , 2),# kernel =2 , stride =2

            # 2nd convolution layer
            nn.Conv2d(32 ,64 ,kernel_size=3 , padding=1), # input channels from last , no of kernels applied , kernel_size , padding 
            nn.ReLU(),
            nn.MaxPool2d(2 , 2),# kernel =2 , stride =2

            # 3rd convolution layer
            nn.Conv2d(64 ,128 ,kernel_size=3 , padding=1), # input channels from last , no of kernels applied , kernel_size , padding 
            nn.ReLU(),
            nn.MaxPool2d(2 , 2),# kernel =2 , stride =2 maxpool reduces the 
        )
        self.fc_layers = nn.Sequential(
            nn.Linear(4*4*128 , 256),
            nn.ReLU(),

            nn.Linear(256 , 10) # this is where our final output 
        )

    def forward(self , x):
         x = self.conv_layers(x)
         x = x.view(x.size(0),-1) # flattening
         x = self.fc_layers(x)

         return x

In [8]:
model = CNN()


In [9]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

# Training the CNN

In [10]:
epochs = 10

for epoch in range(epochs):
    epoch_training_loss = 0.0

    for images , labels in trainloader :
        optimizer.zero_grad()

        output = model.forward(images) #FP
        loss = criterion(output , labels) # loss fxn
        loss.backward() # BP
        optimizer.step() #update params

        epoch_training_loss += loss.item()

    print(f"epoch={epoch+1}/{epochs} & loss={epoch_training_loss /len(trainloader)}")

epoch=1/10 & loss=1.3833375477882297
epoch=2/10 & loss=0.9463172471889144
epoch=3/10 & loss=0.7636386790238988
epoch=4/10 & loss=0.6372412602081323
epoch=5/10 & loss=0.5351021086697079
epoch=6/10 & loss=0.4441737506700599
epoch=7/10 & loss=0.3578727014000763
epoch=8/10 & loss=0.2806778115308498
epoch=9/10 & loss=0.22627042268243286
epoch=10/10 & loss=0.17767265388299056


# Evaluation

In [12]:
correct_labels = 0
total_labels = 0

model.eval()

with torch.no_grad():
    for images ,labels in testloader:
        outputs = model.forward(images)
        _, predicted = torch.max(outputs , 1)
        
        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)
        
print(f"accuracy = {correct_labels / total_labels * 100 }")

accuracy = 75.18


In [14]:
# mam asekd to validation on my your for that create a validationloader from CIFAR or use the testloader it self
# If you only loaded CIFAR-10 like this:

# trainloader = DataLoader(trainset, ...)
# testloader = DataLoader(testset, ...)

# then there is no validation set. You have only:

# Training set
# Test set

# In that case you either:

# use testloader temporarily for evaluation, or
# split the training set into train + validation (recommended).